# Assignment 2

In this assigment, we will work with the *Adult* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/2/adult). Extract the data files into the subdirectory: `../05_src/data/adult/` (relative to `./05_src/`).

# Load the data

Assuming that the files `adult.data` and `adult.test` are in `../05_src/data/adult/`, then you can use the code below to load them.

In [1]:
import pandas as pd
columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
    'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week',
    'native-country', 'income'
]
adult_dt = (pd.read_csv('C:/Users/mid00/Demo_DSI/production/05_src/data/adult/adult.data', header = None, names = columns) #used absolute path
              .assign(income = lambda x: (x.income.str.strip() == '>50K')*1))


# Get X and Y

Create the features data frame and target data:

+ Create a dataframe `X` that holds the features (all columns that are not `income`).
+ Create a dataframe `Y` that holds the target data (`income`).
+ From `X` and `Y`, obtain the training and testing data sets:

    - Use a train-test split of 70-30%. 
    - Set the random state of the splitting function to 42.

In [2]:
#import train_test_split to split data into training and test data sets
from sklearn.model_selection import train_test_split

In [4]:
#Create a dataframe X that holds the features (all columns that are not income)
X = adult_dt.drop(columns =['income'])

#Create a dataframe Y that holds the target data (income).
Y = adult_dt['income']

#perform 70/30 data split : x = features  y= target variable and set random state = 42
X_train, X_test, Y_train, Y_test = train_test_split(X,Y,test_size = 0.3, random_state=42)

print("Training set size :", X_train.shape, Y_train.shape)
print("Testing set size:", Y_train.shape, Y_test.shape)

Training set size : (22792, 14) (22792,)
Testing set size: (22792,) (9769,)


## Random States

Please comment: 

+ What is the [random state](https://scikit-learn.org/stable/glossary.html#term-random_state) of the [splitting function](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)? 
+ Why is it [useful](https://en.wikipedia.org/wiki/Reproducibility)?

*(Comment here.)*

Random state helps to make the results reproducible, which means, it ensures the data split is the same every time the code runs. This is especially useful for data scientists or developers when you collaborate with others, as everyone will have the same data split and can replicate issues and analyze model performance under the same conditions.

# Preprocessing

Create a [Column Transformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) that treats the features as follows:

- Numerical variables

    * Apply [KNN-based imputation for completing missing values](https://scikit-learn.org/stable/modules/generated/sklearn.impute.KNNImputer.html):
        
        + Consider the 7 nearest neighbours.
        + Weight each neighbour by the inverse of its distance, causing closer neigbours to have more influence than more distant ones.
    * [Scale features using statistics that are robust to outliers](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html#sklearn.preprocessing.RobustScaler).

- Categorical variables: 
    
    * Apply a [simple imputation strategy](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html#sklearn.impute.SimpleImputer):

        + Use the most frequent value to complete missing values, also called the *mode*.

    * Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html):
        
        + Handle unknown labels if they exist.
        + Drop one column for binary variables.
    
    
The column transformer should look like this:

![](./images/assignment_2__column_transformer.png)

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer,SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt 

In [6]:
#define numerical transformation considering 7 nearest neighbours and weight by distance
num_transforms =  Pipeline(steps=[
    ('knn_imputer', KNNImputer(n_neighbors=7, weights='distance')),
    ('scaler', RobustScaler())
    ]) 

In [7]:
#define categorical transformation - Simple Imputer using the mode and onehot encoder application
cat_transforms = Pipeline(steps=[
    ('simple_imputer', SimpleImputer(strategy='most_frequent')),  
    ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist'))  
])


In [8]:
#display data types for each column in the dataframe
print(adult_dt.dtypes)

age                int64
workclass         object
fnlwgt             int64
education         object
education-num      int64
marital-status    object
occupation        object
relationship      object
race              object
sex               object
capital-gain       int64
capital-loss       int64
hours-per-week     int64
native-country    object
income             int32
dtype: object


In [19]:
#identify the numerical and categorical features in the dataframe and put them in a separate list
numerical_columns = adult_dt.select_dtypes(include=['int64']).columns.tolist() #excluded int32 datatype as it indicates 'income' column which was excluded in X_train

categorical_columns = adult_dt.select_dtypes(include=['object']).columns.tolist()

print ("Numerical columns in the dataframe adult_dt are:", numerical_columns)
print ("Categorical columns in the dataframe adult_dt are:", categorical_columns)

Numerical columns in the dataframe adult_dt are: ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
Categorical columns in the dataframe adult_dt are: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']


In [20]:
# Create the ColumnTransformer and drop any columns not specified
preprocessor = ColumnTransformer(
    transformers=[
        ('num_transforms', num_transforms, numerical_columns),  
        ('cat_transforms', cat_transforms, categorical_columns)  
    ],
    remainder='drop'  
)
print("\nColumnTransformer structure:")
preprocessor


ColumnTransformer structure:


ColumnTransformer(transformers=[('num_transforms',
                                 Pipeline(steps=[('knn_imputer',
                                                  KNNImputer(n_neighbors=7,
                                                             weights='distance')),
                                                 ('scaler', RobustScaler())]),
                                 ['age', 'fnlwgt', 'education-num',
                                  'capital-gain', 'capital-loss',
                                  'hours-per-week']),
                                ('cat_transforms',
                                 Pipeline(steps=[('simple_imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist'))]),
                                 ['workclass', 'education', 'marital-status',
                                  'occupation', 'relationship', 'race', 'sex',
                                  'native-country'])])

## Model Pipeline

Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `classifier` and assign a [`RandomForestClassifier()`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) to it.

The pipeline looks like this:

![](./images/assignment_2__pipeline.png)

In [21]:
from sklearn.ensemble import RandomForestClassifier

In [22]:
#create pipeline and apply preprocessing to the data and use a RandomForest classifier
model_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('classifier', RandomForestClassifier())
])

print("Model Pipeline:")
model_pipeline

Model Pipeline:


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num_transforms',
                                                  Pipeline(steps=[('knn_imputer',
                                                                   KNNImputer(n_neighbors=7,
                                                                              weights='distance')),
                                                                  ('scaler',
                                                                   RobustScaler())]),
                                                  ['age', 'fnlwgt',
                                                   'education-num',
                                                   'capital-gain',
                                                   'capital-loss',
                                                   'hours-per-week']),
                                                 ('cat_transforms',
                                                  Pipeline(steps=[('simple_imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='infrequent_if_exist'))]),
                                                  ['workclass', 'education',
                                                   'marital-status',
                                                   'occupation', 'relationship',
                                                   'race', 'sex',
                                                   'native-country'])])),
                ('classifier', RandomForestClassifier())])

# Cross-Validation

Evaluate the model pipeline using [`cross_validate()`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html):

+ Measure the following [preformance metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#common-cases-predefined-values): negative log loss, ROC AUC, accuracy, and balanced accuracy.
+ Report the training and validation results. 
+ Use five folds.


Display the fold-level results as a pandas data frame and sorted by negative log loss of the test (validation) set.

In [23]:
from sklearn.model_selection import cross_validate
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score, balanced_accuracy_score

In [16]:
print(X_train.columns)

Index(['age', 'workclass', 'fnlwgt', 'education', 'education-num',
       'marital-status', 'occupation', 'relationship', 'race', 'sex',
       'capital-gain', 'capital-loss', 'hours-per-week', 'native-country'],
      dtype='object')


In [33]:
#define scoring metrics : define dictionary(scoring) with the performance metrics required to evaluate during cross-validation. 
scoring = {
    'neg_log_loss': 'neg_log_loss',
    'roc_auc': 'roc_auc',
    'accuracy': 'accuracy',
    'balanced_accuracy': 'balanced_accuracy'
}

#cross validate training data for 5 folds
validate_results = cross_validate(model_pipeline, X_train, Y_train, cv=5, scoring=scoring, return_train_score=True)

#result of cross validation is stored in validate_results
validate_results

{'fit_time': array([9.15475035, 8.71863961, 8.49134469, 8.74164462, 8.70003223]),
 'score_time': array([0.16559315, 0.1606214 , 0.17051148, 0.16324306, 0.16939807]),
 'test_neg_log_loss': array([-0.34926672, -0.37354889, -0.37423638, -0.36486539, -0.39692399]),
 'train_neg_log_loss': array([-0.08122931, -0.08098176, -0.08159525, -0.08224873, -0.08161385]),
 'test_roc_auc': array([0.90466908, 0.90298527, 0.90239938, 0.90667647, 0.90094468]),
 'train_roc_auc': array([1., 1., 1., 1., 1.]),
 'test_accuracy': array([0.85106383, 0.84755429, 0.85344449, 0.85849057, 0.85870996]),
 'train_accuracy': array([1.        , 1.        , 0.99994516, 1.        , 1.        ]),
 'test_balanced_accuracy': array([0.77723213, 0.77061341, 0.77496701, 0.78260507, 0.78028837]),
 'train_balanced_accuracy': array([1.        , 1.        , 0.99988693, 1.        , 1.        ])}

In [31]:

# Create a DataFrame from the cross-validation results
df_results = pd.DataFrame({
    'Fold': range(1, 6),  # Folds are indexed from 1 to 5
    'Train Neg Log Loss': validate_results['train_neg_log_loss'],
    'Test Neg Log Loss': validate_results['test_neg_log_loss'],
    'Train ROC AUC': validate_results['train_roc_auc'],
    'Test ROC AUC': validate_results['test_roc_auc'],
    'Train Accuracy': validate_results['train_accuracy'],
    'Test Accuracy': validate_results['test_accuracy'],
    'Train Balanced Accuracy': validate_results['train_balanced_accuracy'],
    'Test Balanced Accuracy': validate_results['test_balanced_accuracy']
})

# Sort the DataFrame by Test Neg Log Loss
df_results_sorted = df_results.sort_values(by='Test Neg Log Loss')

# Display the sorted DataFrame
print(df_results_sorted)

   Fold  Train Neg Log Loss  Test Neg Log Loss  Train ROC AUC  Test ROC AUC  \
4     5           -0.081426          -0.400900            1.0      0.902216   
2     3           -0.081128          -0.375730            1.0      0.901294   
0     1           -0.082004          -0.365653            1.0      0.903770   
1     2           -0.081254          -0.360109            1.0      0.902888   
3     4           -0.082007          -0.359208            1.0      0.905595   

   Train Accuracy  Test Accuracy  Train Balanced Accuracy  \
4        0.999945       0.857394                 0.999887   
2        1.000000       0.852567                 1.000000   
0        0.999945       0.850844                 0.999887   
1        1.000000       0.844922                 1.000000   
3        1.000000       0.856955                 1.000000   

   Test Balanced Accuracy  
4                0.776650  
2                0.774080  
0                0.776780  
1                0.766418  
3                0

Calculate the mean of each metric. 

In [34]:
#to compute mean import numpy library
import numpy as np  

#define all metrics for which the mean needs to be calculated
metric_list = ['neg_log_loss', 'roc_auc', 'accuracy', 'balanced_accuracy']

#calculate the mean of each performance metric and store in a dictionary
mean_results = {f'Mean {metric}': {
    'Train': np.mean(validate_results[f'train_{metric}']),
    'Test': np.mean(validate_results[f'test_{metric}'])
} for metric in metric_list}

print (mean_results)



{'Mean neg_log_loss': {'Train': -0.08153377884200824, 'Test': -0.371768274359529}, 'Mean roc_auc': {'Train': 1.0, 'Test': 0.9035349768398566}, 'Mean accuracy': {'Train': 0.9999890314796535, 'Test': 0.8538526275507674}, 'Mean balanced_accuracy': {'Train': 0.9999773857982813, 'Test': 0.7771411987126898}}


Calculate the same performance metrics (negative log loss, ROC AUC, accuracy, and balanced accuracy) using the testing data `X_test` and `Y_test`. Display results as a dictionary.

*Tip*: both, `roc_auc()` and `neg_log_loss()` will require prediction scores from `pipe.predict_proba()`. However, for `roc_auc()` you should only pass the last column `Y_pred_proba[:, 1]`. Use `Y_pred_proba` with `neg_log_loss()`.

In [37]:
#fit the model with training dataset
model_pipeline.fit(X_train, Y_train)


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num_transforms',
                                                  Pipeline(steps=[('knn_imputer',
                                                                   KNNImputer(n_neighbors=7,
                                                                              weights='distance')),
                                                                  ('scaler',
                                                                   RobustScaler())]),
                                                  ['age', 'fnlwgt',
                                                   'education-num',
                                                   'capital-gain',
                                                   'capital-loss',
                                                   'hours-per-week']),
                                                 ('cat_transforms',
                                                  Pipeline(steps=[('simple_imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='infrequent_if_exist'))]),
                                                  ['workclass', 'education',
                                                   'marital-status',
                                                   'occupation', 'relationship',
                                                   'race', 'sex',
                                                   'native-country'])])),
                ('classifier', RandomForestClassifier())])

In [38]:

# Get the prediction probabilities from the model using X_test
Y_pred_proba = model_pipeline.predict_proba(X_test)

# calculate performance metrics 
neg_log_loss_value = log_loss(Y_test, Y_pred_proba)
roc_auc_value = roc_auc_score(Y_test, Y_pred_proba[:, 1])
Y_pred = model_pipeline.predict(X_test)  
accuracy_value = accuracy_score(Y_test, Y_pred)
balanced_accuracy_value = balanced_accuracy_score(Y_test, Y_pred)

# Store the results in a dictionary
test_results = {
    'Negative Log Loss': neg_log_loss_value,
    'ROC AUC': roc_auc_value,
    'Accuracy': accuracy_value,
    'Balanced Accuracy': balanced_accuracy_value
}
print(test_results)

{'Negative Log Loss': 0.38479903787787306, 'ROC AUC': 0.899593817587171, 'Accuracy': 0.8556658818712253, 'Balanced Accuracy': 0.7749024831791091}


# Target Recoding

In the first code chunk of this document, we loaded the data and immediately recoded the target variable `income`. Why is this [convenient](https://scikit-learn.org/stable/modules/model_evaluation.html#binary-case)?

The specific line was:

```
adult_dt = (pd.read_csv('../05_src/data/adult/adult.data', header = None, names = columns)
              .assign(income = lambda x: (x.income.str.strip() == '>50K')*1))
```

(Answer here.)

In the original dataset, 'income' was in a categorical format with the values <=50K or >50K and it was converted into numerical values 0 and 1. With encoding the model can interpret the data correctly and improves performance. 

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Becker,Barry and Kohavi,Ronny. (1996). Adult. UCI Machine Learning Repository. https://doi.org/10.24432/C5XW20.